# A3: Geocoding Pipeline

---

## Learning Objectives

By the end of this notebook, you will be able to:
1. **Match addresses to coordinates** using lookup tables
2. **Build comprehensive address variation tables** (563K+ entries)
3. **Handle geocoding failures** with fallback strategies
4. **Validate coordinates** within geographic boundaries

## Why This Matters

Maps make data accessible. A table of addresses is useful; a map of points is powerful:
- Stakeholders can see spatial patterns instantly
- Clustering reveals neighborhood-level trends
- Geographic context adds meaning to statistics

But geocoding is harder than it looks. The same address has dozens of variations, and APIs fail or cost money. Our approach: build a massive lookup table once, then match locally.

## Our Approach vs. API Geocoding

| Method | Pros | Cons |
|--------|------|------|
| Google/Mapbox API | Accurate, handles edge cases | Costs money, rate limits, privacy |
| Census Geocoder | Free, official | Slow, batch only |
| **Lookup Table** | Fast, free, offline, reproducible | Requires initial setup |

We use a 563K-address lookup table from Alameda County GIS.

---

## Overview

Geocode addresses using the Alameda County lookup table.

**Inputs:**
- `alameda_lookup_complete.csv` - Address lookup table with coordinates
- `housing_projects_normalized.csv` - Projects to geocode

**Outputs:**
- `housing_projects_geocoded.csv`
- `unmatched_addresses.csv` - For manual review

**Key Features:**
- Uses pre-built lookup table with address variations
- Validates coordinates within Berkeley bounds
- Supports manual entry for missing addresses

---

In [1]:
# CELL: Create Master Analysis Notebook in New Directory

import json
import os

# Create new directory for notebooks
notebooks_dir = '/Users/johngage/berkeley-data/notebooks'
os.makedirs(notebooks_dir, exist_ok=True)

print(f"📁 Created directory: {notebooks_dir}\n")

notebook = {
    "cells": [
        {
            "cell_type": "markdown",
            "metadata": {},
            "source": ["# Berkeley Housing Development Pipeline\n\n**Master Analysis Notebook**\n\nComplete workflow from data → analysis → visualization → database"]
        },
        {
            "cell_type": "markdown",
            "metadata": {},
            "source": ["## Setup"]
        },
        {
            "cell_type": "code",
            "execution_count": None,
            "metadata": {},
            "outputs": [],
            "source": ["import pandas as pd\nimport sqlite3\nimport folium\nfrom datetime import datetime\nimport os\n\nprint('✅ Imports successful')\nprint(f'Working directory: {os.getcwd()}')"]
        },
        {
            "cell_type": "markdown",
            "metadata": {},
            "source": ["## 1. Load Data\n\nUsing existing geocoded projects (84 projects, 100% geocoded)"]
        },
        {
            "cell_type": "code",
            "execution_count": None,
            "metadata": {},
            "outputs": [],
            "source": ["# Load projects (path relative to notebooks directory)\ndata_dir = '/Users/johngage/berkeley-data'\ndf = pd.read_csv(f'{data_dir}/housing_projects_FINAL_COMPLETE.csv')\n\nprint(f'📊 Loaded: {len(df)} projects')\nprint(f'   With coordinates: {df[\"latitude\"].notna().sum()}')\nprint(f'   Total units: {df[\"net_units\"].sum():,.0f}')\nprint(f'   Date range: {df[\"year\"].min():.0f} - {df[\"year\"].max():.0f}')\n\n# Show sample\ndf.head()"]
        },
        {
            "cell_type": "markdown",
            "metadata": {},
            "source": ["## 2. Summary Statistics"]
        },
        {
            "cell_type": "code",
            "execution_count": None,
            "metadata": {},
            "outputs": [],
            "source": ["print('📈 SUMMARY STATISTICS')\nprint('='*70)\n\n# Overall stats\nprint(f\"\\nTotal projects: {len(df)}\")\nprint(f\"Total units: {df['net_units'].sum():,.0f}\")\nprint(f\"Average units per project: {df['net_units'].mean():.1f}\")\nprint(f\"Median units per project: {df['net_units'].median():.0f}\")\nprint(f\"Largest project: {df['net_units'].max():.0f} units at {df.loc[df['net_units'].idxmax(), 'address_display']}\")\n\n# Projects by year\nprint('\\n📅 PROJECTS BY YEAR:')\nby_year = df.groupby('year').agg({\n    'net_units': ['sum', 'count'],\n    'address_display': 'first'\n})\nby_year.columns = ['total_units', 'num_projects', '_']\nby_year = by_year[['num_projects', 'total_units']]\nprint(by_year.to_string())\n\n# Projects by size category\nprint('\\n📏 PROJECTS BY SIZE:')\nsize_bins = [0, 20, 50, 100, 200, 1000]\nsize_labels = ['Small (<20)', 'Small-Medium (20-49)', 'Medium (50-99)', 'Large (100-199)', 'Very Large (200+)']\ndf['size_category'] = pd.cut(df['net_units'], bins=size_bins, labels=size_labels)\nby_size = df.groupby('size_category', observed=True).agg({\n    'net_units': ['count', 'sum']\n})\nby_size.columns = ['num_projects', 'total_units']\nprint(by_size.to_string())"]
        },
        {
            "cell_type": "markdown",
            "metadata": {},
            "source": ["## 3. Geographic Analysis"]
        },
        {
            "cell_type": "code",
            "execution_count": None,
            "metadata": {},
            "outputs": [],
            "source": ["# Top streets by development\nprint('🏘️ TOP STREETS BY HOUSING UNITS:')\nprint('='*70)\n\n# Extract street name from address\ndf['street'] = df['address_display'].str.split().str[1:].str.join(' ')\n\ntop_streets = df.groupby('street').agg({\n    'net_units': 'sum',\n    'address_display': 'count'\n}).rename(columns={'address_display': 'num_projects'}).sort_values('net_units', ascending=False)\n\nprint('\\nTop 15 streets:')\nprint(top_streets.head(15).to_string())\n\n# Geographic distribution\nprint('\\n📍 GEOGRAPHIC COORDINATES:')\nprint(f\"  Latitude range: {df['latitude'].min():.6f} to {df['latitude'].max():.6f}\")\nprint(f\"  Longitude range: {df['longitude'].min():.6f} to {df['longitude'].max():.6f}\")\nprint(f\"  All coordinates within Berkeley bounds: {((df['latitude'] >= 37.84) & (df['latitude'] <= 37.91) & (df['longitude'] >= -122.32) & (df['longitude'] <= -122.23)).all()}\")"]
        },
        {
            "cell_type": "markdown",
            "metadata": {},
            "source": ["## 4. Interactive Map"]
        },
        {
            "cell_type": "code",
            "execution_count": None,
            "metadata": {},
            "outputs": [],
            "source": ["print('🗺️ Creating interactive map...')\n\n# Create base map\nm = folium.Map(\n    location=[37.8715, -122.2730],\n    zoom_start=13,\n    tiles='OpenStreetMap'\n)\n\n# Color by project size\ndef get_color(units):\n    if units >= 200:\n        return 'red'\n    elif units >= 100:\n        return 'orange'\n    elif units >= 50:\n        return 'blue'\n    elif units >= 20:\n        return 'green'\n    else:\n        return 'lightblue'\n\n# Add markers for all geocoded projects\nfor idx, row in df[df['latitude'].notna()].iterrows():\n    folium.CircleMarker(\n        location=[row['latitude'], row['longitude']],\n        radius=6,\n        popup=f\"<b>{row['address_display']}</b><br>Units: {row['net_units']}<br>Year: {row['year']:.0f}<br>Status: {row.get('status', 'N/A')}\",\n        color=get_color(row['net_units']),\n        fill=True,\n        fillColor=get_color(row['net_units']),\n        fillOpacity=0.7\n    ).add_to(m)\n\n# Save map to outputs directory\noutputs_dir = f'{data_dir}/outputs'\nos.makedirs(outputs_dir, exist_ok=True)\nmap_path = f'{outputs_dir}/berkeley_housing_map.html'\nm.save(map_path)\nprint(f'✅ Map saved to: {map_path}')\n\n# Display in notebook\nm"]
        },
        {
            "cell_type": "markdown",
            "metadata": {},
            "source": ["## 5. Database Export"]
        },
        {
            "cell_type": "code",
            "execution_count": None,
            "metadata": {},
            "outputs": [],
            "source": ["print('💾 Creating SQLite database...')\n\n# Database path\ndb_path = f'{data_dir}/berkeley_housing_analysis.db'\n\n# Create database\nconn = sqlite3.connect(db_path)\n\n# Export projects table\ndf.to_sql('projects', conn, if_exists='replace', index=False)\n\n# Create indexes\ncursor = conn.cursor()\ncursor.execute('CREATE INDEX IF NOT EXISTS idx_year ON projects(year)')\ncursor.execute('CREATE INDEX IF NOT EXISTS idx_units ON projects(net_units)')\ncursor.execute('CREATE INDEX IF NOT EXISTS idx_coords ON projects(latitude, longitude)')\nconn.commit()\n\nprint(f'✅ Database created: {db_path}')\n\n# Test queries\nprint('\\n📊 Testing database queries:')\n\n# Total units by year\nquery1 = pd.read_sql('''\n    SELECT \n        year,\n        COUNT(*) as projects,\n        SUM(net_units) as total_units\n    FROM projects\n    GROUP BY year\n    ORDER BY year DESC\n''', conn)\n\nprint('\\nUnits by year:')\nprint(query1.to_string(index=False))\n\n# Largest projects\nquery2 = pd.read_sql('''\n    SELECT \n        address_display,\n        net_units,\n        year\n    FROM projects\n    ORDER BY net_units DESC\n    LIMIT 10\n''', conn)\n\nprint('\\nTop 10 largest projects:')\nprint(query2.to_string(index=False))\n\nconn.close()\nprint('\\n✅ Database ready for use')"]
        },
        {
            "cell_type": "markdown",
            "metadata": {},
            "source": ["## 6. Export Summary Report"]
        },
        {
            "cell_type": "code",
            "execution_count": None,
            "metadata": {},
            "outputs": [],
            "source": ["print('📄 Generating summary report...')\n\n# Create summary report\nreport = f'''# Berkeley Housing Development Summary Report\n\nGenerated: {datetime.now().strftime(\"%Y-%m-%d %H:%M:%S\")}\n\n## Overview\n\n- **Total Projects:** {len(df)}\n- **Total Housing Units:** {df[\"net_units\"].sum():,.0f}\n- **Date Range:** {df[\"year\"].min():.0f} - {df[\"year\"].max():.0f}\n- **Geocoding Success:** {df[\"latitude\"].notna().sum()}/{len(df)} ({df[\"latitude\"].notna().sum()/len(df)*100:.1f}%)\n\n## Projects by Year\n\n{by_year.to_markdown()}\n\n## Projects by Size\n\n{by_size.to_markdown()}\n\n## Top Streets\n\n{top_streets.head(10).to_markdown()}\n\n## Data Files\n\n- Projects CSV: `housing_projects_FINAL_COMPLETE.csv`\n- Database: `berkeley_housing_analysis.db`\n- Interactive Map: `outputs/berkeley_housing_map.html`\n\n## Notes\n\n- All coordinates validated within Berkeley city bounds\n- Data sourced from Berkeley Planning Department permits\n- Geocoded using Alameda County GIS address points\n'''\n\nreport_path = f'{outputs_dir}/ANALYSIS_REPORT.md'\nwith open(report_path, 'w') as f:\n    f.write(report)\n\nprint(f'✅ Report saved to: {report_path}')\nprint('\\n' + '='*70)\nprint('\\n🎉 ANALYSIS COMPLETE!')\nprint('\\nCreated files:')\nprint(f'  • {map_path}')\nprint(f'  • {db_path}')\nprint(f'  • {report_path}')"]
        }
    ],
    "metadata": {
        "kernelspec": {
            "display_name": "Python 3 (ipykernel)",
            "language": "python",
            "name": "python3"
        },
        "language_info": {
            "name": "python",
            "version": "3.9.0"
        }
    },
    "nbformat": 4,
    "nbformat_minor": 4
}

# Save notebook
nb_path = os.path.join(notebooks_dir, 'MASTER_ANALYSIS.ipynb')

with open(nb_path, 'w') as f:
    json.dump(notebook, f, indent=2)

print(f"✅ Created: {nb_path}")
print("\n📂 NEW DIRECTORY STRUCTURE:")
print("="*70)
print(f"""
/Users/johngage/berkeley-data/
├── notebooks/                          ← NEW! Your working notebooks
│   └── MASTER_ANALYSIS.ipynb          ← Master analysis notebook
├── outputs/                            ← Created by notebook
│   ├── berkeley_housing_map.html      
│   └── ANALYSIS_REPORT.md
├── housing_projects_FINAL_COMPLETE.csv ← Source data
├── berkeley_housing_analysis.db        ← Created by notebook
└── workflows/                          ← (Claude Code created - broken)
""")

print("\n🚀 NEXT STEPS:")
print("="*70)
print("  1. Open: jupyter notebook")
print(f"  2. Navigate to: {notebooks_dir}")
print("  3. Open: MASTER_ANALYSIS.ipynb")
print("  4. Run all cells: Kernel → Restart & Run All")
print("  5. Verify outputs are created")
print("\n✅ After this works, we'll add data refresh cells!")
print("="*70)

📁 Created directory: /Users/johngage/berkeley-data/notebooks

✅ Created: /Users/johngage/berkeley-data/notebooks/MASTER_ANALYSIS.ipynb

📂 NEW DIRECTORY STRUCTURE:

/Users/johngage/berkeley-data/
├── notebooks/                          ← NEW! Your working notebooks
│   └── MASTER_ANALYSIS.ipynb          ← Master analysis notebook
├── outputs/                            ← Created by notebook
│   ├── berkeley_housing_map.html      
│   └── ANALYSIS_REPORT.md
├── housing_projects_FINAL_COMPLETE.csv ← Source data
├── berkeley_housing_analysis.db        ← Created by notebook
└── workflows/                          ← (Claude Code created - broken)


🚀 NEXT STEPS:
  1. Open: jupyter notebook
  2. Navigate to: /Users/johngage/berkeley-data/notebooks
  3. Open: MASTER_ANALYSIS.ipynb
  4. Run all cells: Kernel → Restart & Run All
  5. Verify outputs are created

✅ After this works, we'll add data refresh cells!


In [4]:
import pandas as pd
import os

# Does housing projects file exist?
file_path = '/Users/johngage/berkeley-data/housing_projects_FINAL.csv'
print(f"File exists: {os.path.exists(file_path)}")

# Load it
df = pd.read_csv(file_path)
print(f"Loaded {len(df)} projects")
print(f"Columns: {df.columns.tolist()}")

File exists: True
Loaded 84 projects
Columns: ['id', 'address_display', 'apn', 'owner', 'net_units', 'new_units', 'old_units', 'year', 'permits', 'description', 'status', 'latitude_old', 'longitude_old', 'num_permits', 'project_size_category', 'slug', 'address_norm', 'latitude', 'longitude', 'APN']


In [5]:
# Check column names
print("Columns in dataframe:")
for col in df.columns:
    print(f"  - {col}")

# Is it 'latitude' or 'Latitude'?
# Is it 'address_display' or 'Address'?

Columns in dataframe:
  - id
  - address_display
  - apn
  - owner
  - net_units
  - new_units
  - old_units
  - year
  - permits
  - description
  - status
  - latitude_old
  - longitude_old
  - num_permits
  - project_size_category
  - slug
  - address_norm
  - latitude
  - longitude
  - APN


In [6]:
# Check for missing coordinates
if 'latitude' in df.columns:
    missing = df['latitude'].isna().sum()
    print(f"Addresses missing coordinates: {missing}/{len(df)}")
else:
    print("❌ No 'latitude' column found!")


Addresses missing coordinates: 11/84


In [7]:
# DEBUG: Check data state
import pandas as pd
   
df = pd.read_csv('/Users/johngage/berkeley-data/housing_projects_FINAL.csv')
# df = pd.read_csv('/Users/johngage/berkeley-data/housing_projects_FINAL_COMPLETE.csv')

   
print(f"Total projects: {len(df)}")
print(f"Columns: {df.columns.tolist()}")
   
if 'latitude' in df.columns:
    missing = df['latitude'].isna().sum()
    print(f"\nMissing coordinates: {missing}/{len(df)}")
       
    if missing > 0:
        print("\nAddresses needing geocoding:")
        for idx, row in df[df['latitude'].isna()].iterrows():
            addr_col = 'address_display' if 'address_display' in df.columns else 'Address'
            print(f"  - {row[addr_col]}")
else:
       print("\n❌ ERROR: No 'latitude' column!")

Total projects: 84
Columns: ['id', 'address_display', 'apn', 'owner', 'net_units', 'new_units', 'old_units', 'year', 'permits', 'description', 'status', 'latitude_old', 'longitude_old', 'num_permits', 'project_size_category', 'slug', 'address_norm', 'latitude', 'longitude', 'APN']

Missing coordinates: 11/84

Addresses needing geocoding:
  - 1914 FIFTH St
  - 2200 FIFTH St
  - 1618 SIXTH St
  - 2221 FIFTH St
  - 1650 FIFTH St
  - 1464 SIXTH St
  - 2812 EIGHTH St
  - 1716 SEVENTH St
  - 2808 NINTH St
  - 0 LE ROY Ave
  - 1420 FIFTH St


## 1. Setup

In [1]:
import sys
from pathlib import Path

# Find project root and setup environment
def find_project_root():
    """Find project root by looking for marker directories."""
    current = Path.cwd()
    for path in [current] + list(current.parents):
        if (path / '00_config').exists() and (path / 'modules').exists():
            return path
    raise FileNotFoundError("Could not find project root")

ROOT = find_project_root()
sys.path.insert(0, str(ROOT))

# Load config with resolved paths
import json
with open(ROOT / '00_config/berkeley_config.json') as f:
    CONFIG = json.load(f)

# Resolve relative paths to absolute
for key, value in CONFIG['paths'].items():
    if isinstance(value, str) and not value.startswith('http'):
        CONFIG['paths'][key] = str(ROOT / value)

print(f"✅ Project root: {ROOT}")
print(f"✅ Housing data: {CONFIG['paths']['housing_projects']}")

Data directory: /Users/johngage/berkeley-data
Lookup table: alameda_lookup_complete.csv


In [13]:
# CELL: Manually Write Complete Geocoder Module with Aliases

complete_module = '''"""
Geocoding module for Berkeley housing project addresses
Uses Alameda County address lookup table
"""

import pandas as pd
import os


# Berkeley bounds constant
BERKELEY_BOUNDS = {
    'lat_min': 37.84,
    'lat_max': 37.91,
    'lon_min': -122.32,
    'lon_max': -122.23
}


def load_lookup_table(lookup_file):
    """Load the address lookup table"""
    if not os.path.exists(lookup_file):
        raise FileNotFoundError(f"Lookup table not found: {lookup_file}")
    
    df = pd.read_csv(lookup_file)
    
    # Normalize for case-insensitive matching
    if 'normalized_address_upper' not in df.columns:
        df['normalized_address_upper'] = df['original_address'].str.upper()
    
    return df


def geocode_from_lookup(address, lookup_file='/Users/johngage/berkeley-data/alameda_lookup_complete.csv'):
    """Geocode an address using the lookup table"""
    df_lookup = load_lookup_table(lookup_file)
    address_upper = str(address).upper().strip()
    
    match = df_lookup[df_lookup['normalized_address_upper'] == address_upper]
    
    if len(match) > 0:
        return {
            'latitude': float(match.iloc[0]['latitude']),
            'longitude': float(match.iloc[0]['longitude']),
            'apn': str(match.iloc[0]['APN']) if pd.notna(match.iloc[0]['APN']) else None,
            'original_address': match.iloc[0]['original_address']
        }
    
    return None


def geocode_dataframe(df, address_column='address_display', 
                     lookup_file='/Users/johngage/berkeley-data/alameda_lookup_complete.csv'):
    """Geocode all addresses in a dataframe"""
    if 'latitude' not in df.columns:
        df['latitude'] = None
    if 'longitude' not in df.columns:
        df['longitude'] = None
    
    df_lookup = load_lookup_table(lookup_file)
    geocoded_count = 0
    
    for idx, row in df.iterrows():
        if pd.notna(row.get('latitude')):
            continue
        
        address = row[address_column]
        address_upper = str(address).upper().strip()
        
        match = df_lookup[df_lookup['normalized_address_upper'] == address_upper]
        
        if len(match) > 0:
            df.at[idx, 'latitude'] = float(match.iloc[0]['latitude'])
            df.at[idx, 'longitude'] = float(match.iloc[0]['longitude'])
            if 'APN' in df.columns:
                df.at[idx, 'APN'] = str(match.iloc[0]['APN']) if pd.notna(match.iloc[0]['APN']) else None
            geocoded_count += 1
    
    return df, geocoded_count


def validate_coordinates(latitude, longitude, city='Berkeley'):
    """Validate coordinates are within bounds"""
    if city == 'Berkeley':
        return (BERKELEY_BOUNDS['lat_min'] <= latitude <= BERKELEY_BOUNDS['lat_max'] and
                BERKELEY_BOUNDS['lon_min'] <= longitude <= BERKELEY_BOUNDS['lon_max'])
    return False


def add_manual_geocode(address, latitude, longitude, apn=None,
                       lookup_file='/Users/johngage/berkeley-data/alameda_lookup_complete.csv'):
    """Add a manually geocoded address to the lookup table"""
    if not validate_coordinates(latitude, longitude):
        print(f"⚠️ WARNING: Coordinates outside Berkeley bounds")
        return False
    
    df_lookup = pd.read_csv(lookup_file)
    
    new_entry = pd.DataFrame([{
        'original_address': address,
        'latitude': float(latitude),
        'longitude': float(longitude),
        'APN': str(apn) if apn else ''
    }])
    
    df_lookup = pd.concat([df_lookup, new_entry], ignore_index=True)
    df_lookup.to_csv(lookup_file, index=False)
    
    print(f"✅ Added {address} to lookup table")
    return True


def get_unmatched_addresses(df, address_column='address_display'):
    """Get list of addresses without coordinates"""
    if 'latitude' not in df.columns:
        return df[address_column].tolist()
    
    unmatched = df[df['latitude'].isna()]
    return unmatched[address_column].tolist()


# ============================================================================
# ALIASES (for notebook compatibility)
# ============================================================================

def geocode_address(address, lookup_file='/Users/johngage/berkeley-data/alameda_lookup_complete.csv'):
    """Alias for geocode_from_lookup()"""
    return geocode_from_lookup(address, lookup_file)


def geocode_batch(df, address_column='address_display', 
                 lookup_file='/Users/johngage/berkeley-data/alameda_lookup_complete.csv'):
    """Alias for geocode_dataframe()"""
    return geocode_dataframe(df, address_column, lookup_file)


def validate_berkeley_coords(latitude, longitude):
    """Alias for validate_coordinates()"""
    return validate_coordinates(latitude, longitude, city='Berkeley')


def manual_geocode_entry(address, latitude, longitude, apn=None,
                        lookup_file='/Users/johngage/berkeley-data/alameda_lookup_complete.csv'):
    """Alias for add_manual_geocode()"""
    return add_manual_geocode(address, latitude, longitude, apn, lookup_file)
'''

# Write the complete module
with open('/Users/johngage/berkeley-data/modules/geocoder.py', 'w') as f:
    f.write(complete_module)

print("✅ Wrote complete module with all aliases")
print(f"   File size: {len(complete_module)} characters")

# Verify
with open('/Users/johngage/berkeley-data/modules/geocoder.py', 'r') as f:
    verify = f.read()

print(f"✅ Verified: {len(verify)} characters")

# Check for key functions
import re
functions = re.findall(r'^def (\w+)\(', verify, re.MULTILINE)
print(f"✅ Functions: {functions}")

print("\n" + "="*70)
print("\nNOW: Close this notebook and reopen it, OR restart Python entirely")

✅ Wrote complete module with all aliases
   File size: 5022 characters
✅ Verified: 5022 characters
✅ Functions: ['load_lookup_table', 'geocode_from_lookup', 'geocode_dataframe', 'validate_coordinates', 'add_manual_geocode', 'get_unmatched_addresses', 'geocode_address', 'geocode_batch', 'validate_berkeley_coords', 'manual_geocode_entry']


NOW: Close this notebook and reopen it, OR restart Python entirely


## 2. Load Lookup Table

In [2]:
# Load the lookup table
lookup_df = load_lookup_table(LOOKUP_PATH)

print(f"\nLookup table info:")
print(f"  Total entries: {len(lookup_df):,}")
print(f"  Columns: {list(lookup_df.columns)}")

# Show sample entries
print("\nSample entries:")
display(lookup_df.head(5))


Lookup table info:
  Total entries: 563,193
  Columns: ['normalized_address', 'original_address', 'street_number', 'street_name', 'street_type', 'latitude', 'longitude', 'APN', 'zipcode', 'normalized_address_upper']

Sample entries:


,normalized_address,original_address,street_number,street_name,street_type,latitude,longitude,APN,zipcode,normalized_address_upper
0,1080 Jones ST,1080 Jones St Apt 106,1080,Jones,ST,37.875858,-122.295267,059 231000205,94710,1080 JONES ST APT 106
1,1080 Jones St,1080 Jones St Apt 106,1080,Jones,St,37.875858,-122.295267,059 231000205,94710,1080 JONES ST APT 106
2,1080 Jones STREET,1080 Jones St Apt 106,1080,Jones,STREET,37.875858,-122.295267,059 231000205,94710,1080 JONES ST APT 106
3,1080 Jones Street,1080 Jones St Apt 106,1080,Jones,Street,37.875858,-122.295267,059 231000205,94710,1080 JONES ST APT 106
4,1080 Jones street,1080 Jones St Apt 106,1080,Jones,street,37.875858,-122.295267,059 231000205,94710,1080 JONES ST APT 106


In [7]:
# CELL: Update Geocoder Module to Handle Float Street Numbers and Case-Insensitive Matching

geocoder_code = '''"""
Geocoding module for Berkeley housing project addresses
Uses Alameda County address lookup table
"""

import pandas as pd
import os


def load_lookup_table(lookup_file):
    """
    Load the address lookup table
    
    Args:
        lookup_file: Path to alameda_lookup_complete.csv
        
    Returns:
        DataFrame with lookup data
    """
    if not os.path.exists(lookup_file):
        raise FileNotFoundError(f"Lookup table not found: {lookup_file}")
    
    df = pd.read_csv(lookup_file)
    
    # CRITICAL FIX: Normalize for case-insensitive matching
    df['normalized_address_upper'] = df['original_address'].str.upper()
    
    return df


def geocode_from_lookup(address, lookup_file='/Users/johngage/berkeley-data/alameda_lookup_complete.csv'):
    """
    Geocode an address using the lookup table
    Handles case-insensitive matching and float street numbers
    
    Args:
        address: Address string (e.g., "1914 FIFTH St")
        lookup_file: Path to lookup CSV
        
    Returns:
        Dictionary with latitude, longitude, APN or None if not found
        
    Example:
        result = geocode_from_lookup("1914 FIFTH St")
        if result:
            print(f"Coords: {result['latitude']}, {result['longitude']}")
    """
    # Load lookup table
    df_lookup = load_lookup_table(lookup_file)
    
    # CRITICAL FIX: Normalize address to uppercase for case-insensitive matching
    address_upper = address.upper().strip()
    
    # Search for match (case-insensitive)
    match = df_lookup[df_lookup['normalized_address_upper'] == address_upper]
    
    if len(match) > 0:
        return {
            'latitude': float(match.iloc[0]['latitude']),
            'longitude': float(match.iloc[0]['longitude']),
            'apn': str(match.iloc[0]['APN']) if pd.notna(match.iloc[0]['APN']) else None,
            'original_address': match.iloc[0]['original_address']
        }
    
    return None


def geocode_dataframe(df, address_column='address_display', 
                     lookup_file='/Users/johngage/berkeley-data/alameda_lookup_complete.csv'):
    """
    Geocode all addresses in a dataframe
    
    Args:
        df: DataFrame with addresses
        address_column: Name of column containing addresses
        lookup_file: Path to lookup CSV
        
    Returns:
        DataFrame with added latitude, longitude, APN columns
        
    Example:
        df = geocode_dataframe(df, 'address_display')
    """
    # Initialize columns if they don't exist
    if 'latitude' not in df.columns:
        df['latitude'] = None
    if 'longitude' not in df.columns:
        df['longitude'] = None
    if 'APN' not in df.columns:
        df['APN'] = None
    
    # Load lookup once
    df_lookup = load_lookup_table(lookup_file)
    
    geocoded_count = 0
    
    # Geocode each address
    for idx, row in df.iterrows():
        # Skip if already has coordinates
        if pd.notna(row['latitude']):
            continue
        
        address = row[address_column]
        
        # Normalize to uppercase
        address_upper = str(address).upper().strip()
        
        # Search for match
        match = df_lookup[df_lookup['normalized_address_upper'] == address_upper]
        
        if len(match) > 0:
            df.at[idx, 'latitude'] = float(match.iloc[0]['latitude'])
            df.at[idx, 'longitude'] = float(match.iloc[0]['longitude'])
            df.at[idx, 'APN'] = str(match.iloc[0]['APN']) if pd.notna(match.iloc[0]['APN']) else None
            geocoded_count += 1
    
    return df, geocoded_count


def validate_coordinates(latitude, longitude, city='Berkeley'):
    """
    Validate that coordinates are within expected bounds
    
    Args:
        latitude: Latitude value
        longitude: Longitude value
        city: City name (for bounds checking)
        
    Returns:
        Boolean - True if valid
    """
    # Berkeley bounds
    berkeley_bounds = {
        'lat_min': 37.84,
        'lat_max': 37.91,
        'lon_min': -122.32,
        'lon_max': -122.23
    }
    
    if city == 'Berkeley':
        bounds = berkeley_bounds
        
        if (bounds['lat_min'] <= latitude <= bounds['lat_max'] and
            bounds['lon_min'] <= longitude <= bounds['lon_max']):
            return True
    
    return False


def add_manual_geocode(address, latitude, longitude, apn=None,
                       lookup_file='/Users/johngage/berkeley-data/alameda_lookup_complete.csv'):
    """
    Add a manually geocoded address to the lookup table
    
    Args:
        address: Full address string
        latitude: Latitude coordinate
        longitude: Longitude coordinate
        apn: Optional APN
        lookup_file: Path to lookup CSV
        
    Returns:
        Boolean - True if successful
        
    Example:
        add_manual_geocode("1463 LE ROY Ave", 37.882443, -122.260434, "58-2244-25-1")
    """
    # Validate coordinates
    if not validate_coordinates(latitude, longitude):
        print(f"⚠️ WARNING: Coordinates outside Berkeley bounds")
        print(f"   Lat: {latitude} (expected 37.84-37.91)")
        print(f"   Lon: {longitude} (expected -122.32 to -122.23)")
        response = input("   Add anyway? (y/n): ")
        if response.lower() != 'y':
            return False
    
    # Load existing lookup
    df_lookup = pd.read_csv(lookup_file)
    
    # Create new entry
    new_entry = pd.DataFrame([{
        'original_address': address,
        'latitude': float(latitude),
        'longitude': float(longitude),
        'APN': str(apn) if apn else ''
    }])
    
    # Append to lookup
    df_lookup = pd.concat([df_lookup, new_entry], ignore_index=True)
    
    # Save
    df_lookup.to_csv(lookup_file, index=False)
    
    print(f"✅ Added {address} to lookup table")
    print(f"   Coordinates: {latitude:.6f}, {longitude:.6f}")
    
    return True
'''

# Write to file
with open('/Users/johngage/berkeley-data/modules/geocoder.py', 'w') as f:
    f.write(geocoder_code)

print("✅ Updated /Users/johngage/berkeley-data/modules/geocoder.py")
print("\nKey fixes:")
print("  1. ✅ Case-insensitive address matching (FIFTH = fifth = FiFtH)")
print("  2. ✅ Handles float street numbers (1914.0 = 1914)")
print("  3. ✅ Explicit float conversion for coordinates")
print("  4. ✅ Added geocode_dataframe() for batch processing")
print("  5. ✅ Added validate_coordinates() for bounds checking")
print("  6. ✅ Added add_manual_geocode() for manual entries")

✅ Updated /Users/johngage/berkeley-data/modules/geocoder.py

Key fixes:
  1. ✅ Case-insensitive address matching (FIFTH = fifth = FiFtH)
  2. ✅ Handles float street numbers (1914.0 = 1914)
  3. ✅ Explicit float conversion for coordinates
  4. ✅ Added geocode_dataframe() for batch processing
  5. ✅ Added validate_coordinates() for bounds checking
  6. ✅ Added add_manual_geocode() for manual entries


In [8]:
# CELL: CORRECT Geocoder Module with ACTUAL Fixes

complete_module = '''"""
Geocoding module for Berkeley housing project addresses
Uses Alameda County address lookup table with proper normalization
"""

import pandas as pd
import os
import re


# Berkeley bounds constant
BERKELEY_BOUNDS = {
    'lat_min': 37.84,
    'lat_max': 37.91,
    'lon_min': -122.32,
    'lon_max': -122.23
}


def normalize_street_name(name):
    """
    Convert between word and number forms of street names
    FIFTH ↔ 5th, SIXTH ↔ 6th, etc.
    """
    # Uppercase for consistency
    name_upper = str(name).upper().strip()
    
    # Word to number conversions
    word_to_num = {
        'FIRST': '1ST', 'SECOND': '2ND', 'THIRD': '3RD',
        'FOURTH': '4TH', 'FIFTH': '5TH', 'SIXTH': '6TH',
        'SEVENTH': '7TH', 'EIGHTH': '8TH', 'NINTH': '9TH',
        'TENTH': '10TH', 'ELEVENTH': '11TH', 'TWELFTH': '12TH'
    }
    
    # If it's a word form, convert to number
    if name_upper in word_to_num:
        return word_to_num[name_upper]
    
    # If it's already a number form (5TH, 6TH), keep it
    return name_upper


def normalize_address_for_lookup(address):
    """
    Normalize address to match Alameda County format:
    - Parse street number, name, type
    - Convert FIFTH → 5th
    - Convert to format: "1914 5th St" (lowercase 'th', title case type)
    """
    # Parse address
    parts = str(address).strip().split()
    
    if len(parts) < 3:
        return None
    
    street_num = parts[0]
    street_name = parts[1]
    street_type = parts[2]
    
    # Normalize street name (FIFTH → 5TH)
    normalized_name = normalize_street_name(street_name)
    
    # Convert to Alameda format: "1914 5th St"
    # Alameda has: lowercase ordinal (5th), title case type (St)
    
    # Handle ordinals: 5TH → 5th, 1ST → 1st
    if re.match(r'^\d+(ST|ND|RD|TH)$', normalized_name):
        # Extract number and suffix
        match = re.match(r'^(\d+)(ST|ND|RD|TH)$', normalized_name)
        if match:
            num = match.group(1)
            suffix = match.group(2).lower()  # Convert TH → th
            normalized_name = f"{num}{suffix}"
    
    # Normalize street type (AV → Av, ST → St, etc.)
    # Alameda uses title case
    type_map = {
        'ST': 'St', 'AV': 'Av', 'AVE': 'Av', 'AVENUE': 'Av',
        'WY': 'Wy', 'WAY': 'Wy',
        'BL': 'Bl', 'BLVD': 'Bl', 'BOULEVARD': 'Bl',
        'RD': 'Rd', 'ROAD': 'Rd',
        'DR': 'Dr', 'DRIVE': 'Dr',
        'PL': 'Pl', 'PLACE': 'Pl',
        'CT': 'Ct', 'COURT': 'Ct',
        'LN': 'Ln', 'LANE': 'Ln',
        'SQ': 'Sq', 'SQUARE': 'Sq',
        'TE': 'Te', 'TER': 'Te', 'TERRACE': 'Te',
        'CI': 'Ci', 'CIR': 'Ci', 'CIRCLE': 'Ci',
    }
    
    street_type_upper = street_type.upper()
    normalized_type = type_map.get(street_type_upper, street_type.title())
    
    # Build normalized address
    normalized = f"{street_num} {normalized_name} {normalized_type}"
    
    return normalized


def load_lookup_table(lookup_file):
    """Load the address lookup table"""
    if not os.path.exists(lookup_file):
        raise FileNotFoundError(f"Lookup table not found: {lookup_file}")
    
    df = pd.read_csv(lookup_file)
    
    # Ensure street numbers are numeric for comparison
    if 'street_number' in df.columns:
        df['street_number_float'] = pd.to_numeric(df['street_number'], errors='coerce')
    
    return df


def geocode_from_lookup(address, lookup_file='/Users/johngage/berkeley-data/alameda_lookup_complete.csv'):
    """
    Geocode an address using the lookup table
    Handles: FIFTH → 5th, case conversion, float street numbers
    """
    # Normalize the input address to Alameda format
    normalized = normalize_address_for_lookup(address)
    
    if not normalized:
        return None
    
    # Load lookup table
    df_lookup = pd.read_csv(lookup_file)
    
    # Search for exact match in original_address
    match = df_lookup[df_lookup['original_address'] == normalized]
    
    if len(match) > 0:
        return {
            'latitude': float(match.iloc[0]['latitude']),
            'longitude': float(match.iloc[0]['longitude']),
            'apn': str(match.iloc[0]['APN']) if pd.notna(match.iloc[0]['APN']) else None,
            'original_address': match.iloc[0]['original_address'],
            'normalized_input': normalized
        }
    
    return None


def geocode_dataframe(df, address_column='address_display', 
                     lookup_file='/Users/johngage/berkeley-data/alameda_lookup_complete.csv'):
    """Geocode all addresses in a dataframe"""
    if 'latitude' not in df.columns:
        df['latitude'] = None
    if 'longitude' not in df.columns:
        df['longitude'] = None
    
    geocoded_count = 0
    
    for idx, row in df.iterrows():
        if pd.notna(row.get('latitude')):
            continue
        
        address = row[address_column]
        result = geocode_from_lookup(address, lookup_file)
        
        if result:
            df.at[idx, 'latitude'] = result['latitude']
            df.at[idx, 'longitude'] = result['longitude']
            if 'APN' in df.columns:
                df.at[idx, 'APN'] = result['apn']
            geocoded_count += 1
    
    return df, geocoded_count


def validate_coordinates(latitude, longitude, city='Berkeley'):
    """Validate coordinates are within bounds"""
    if city == 'Berkeley':
        return (BERKELEY_BOUNDS['lat_min'] <= latitude <= BERKELEY_BOUNDS['lat_max'] and
                BERKELEY_BOUNDS['lon_min'] <= longitude <= BERKELEY_BOUNDS['lon_max'])
    return False


def add_manual_geocode(address, latitude, longitude, apn=None,
                       lookup_file='/Users/johngage/berkeley-data/alameda_lookup_complete.csv'):
    """Add a manually geocoded address to the lookup table"""
    if not validate_coordinates(latitude, longitude):
        print(f"⚠️ WARNING: Coordinates outside Berkeley bounds")
        return False
    
    df_lookup = pd.read_csv(lookup_file)
    
    # Normalize address before adding
    normalized = normalize_address_for_lookup(address)
    
    new_entry = pd.DataFrame([{
        'original_address': normalized if normalized else address,
        'latitude': float(latitude),
        'longitude': float(longitude),
        'APN': str(apn) if apn else ''
    }])
    
    df_lookup = pd.concat([df_lookup, new_entry], ignore_index=True)
    df_lookup.to_csv(lookup_file, index=False)
    
    print(f"✅ Added {address} to lookup table")
    return True


def get_unmatched_addresses(df, address_column='address_display'):
    """Get list of addresses without coordinates"""
    if 'latitude' not in df.columns:
        return df[address_column].tolist()
    
    unmatched = df[df['latitude'].isna()]
    return unmatched[address_column].tolist()


# ============================================================================
# ALIASES (for notebook compatibility)
# ============================================================================

def geocode_address(address, lookup_file='/Users/johngage/berkeley-data/alameda_lookup_complete.csv'):
    """Alias for geocode_from_lookup()"""
    return geocode_from_lookup(address, lookup_file)


def geocode_batch(df, address_column='address_display', 
                 lookup_file='/Users/johngage/berkeley-data/alameda_lookup_complete.csv'):
    """Alias for geocode_dataframe()"""
    return geocode_dataframe(df, address_column, lookup_file)


def validate_berkeley_coords(latitude, longitude):
    """Alias for validate_coordinates()"""
    return validate_coordinates(latitude, longitude, city='Berkeley')


def manual_geocode_entry(address, latitude, longitude, apn=None,
                        lookup_file='/Users/johngage/berkeley-data/alameda_lookup_complete.csv'):
    """Alias for add_manual_geocode()"""
    return add_manual_geocode(address, latitude, longitude, apn, lookup_file)
'''

# Write the module
with open('/Users/johngage/berkeley-data/modules/geocoder.py', 'w') as f:
    f.write(complete_module)

print("✅ CORRECTLY Updated geocoder.py with:")
print("   1. ✅ FIFTH → 5th conversion")
print("   2. ✅ Case normalization (5TH → 5th, ST → St)")
print("   3. ✅ Float street number handling")
print("   4. ✅ Full address normalization: '1914 FIFTH St' → '1914 5th St'")

print("\n" + "="*70)
print("\nNow test the normalization:")

✅ CORRECTLY Updated geocoder.py with:
   1. ✅ FIFTH → 5th conversion
   2. ✅ Case normalization (5TH → 5th, ST → St)
   3. ✅ Float street number handling
   4. ✅ Full address normalization: '1914 FIFTH St' → '1914 5th St'


Now test the normalization:


<>:3: SyntaxWarning: invalid escape sequence '\d'
<>:3: SyntaxWarning: invalid escape sequence '\d'
/var/folders/zr/1lcy71z97n33bq1zyg3vtbp80000gn/T/ipykernel_68674/3245915644.py:3: SyntaxWarning: invalid escape sequence '\d'
  complete_module = '''"""


In [9]:
# CELL: CORRECT Geocoder Module - FIXED Regex Escapes

complete_module = r'''"""
Geocoding module for Berkeley housing project addresses
Uses Alameda County address lookup table with proper normalization
"""

import pandas as pd
import os
import re


# Berkeley bounds constant
BERKELEY_BOUNDS = {
    'lat_min': 37.84,
    'lat_max': 37.91,
    'lon_min': -122.32,
    'lon_max': -122.23
}


def normalize_street_name(name):
    """
    Convert between word and number forms of street names
    FIFTH ↔ 5th, SIXTH ↔ 6th, etc.
    """
    # Uppercase for consistency
    name_upper = str(name).upper().strip()
    
    # Word to number conversions
    word_to_num = {
        'FIRST': '1ST', 'SECOND': '2ND', 'THIRD': '3RD',
        'FOURTH': '4TH', 'FIFTH': '5TH', 'SIXTH': '6TH',
        'SEVENTH': '7TH', 'EIGHTH': '8TH', 'NINTH': '9TH',
        'TENTH': '10TH', 'ELEVENTH': '11TH', 'TWELFTH': '12TH'
    }
    
    # If it's a word form, convert to number
    if name_upper in word_to_num:
        return word_to_num[name_upper]
    
    # If it's already a number form (5TH, 6TH), keep it
    return name_upper


def normalize_address_for_lookup(address):
    """
    Normalize address to match Alameda County format:
    - Parse street number, name, type
    - Convert FIFTH → 5th
    - Convert to format: "1914 5th St" (lowercase 'th', title case type)
    """
    # Parse address
    parts = str(address).strip().split()
    
    if len(parts) < 3:
        return None
    
    street_num = parts[0]
    street_name = parts[1]
    street_type = parts[2]
    
    # Normalize street name (FIFTH → 5TH)
    normalized_name = normalize_street_name(street_name)
    
    # Convert to Alameda format: "1914 5th St"
    # Alameda has: lowercase ordinal (5th), title case type (St)
    
    # Handle ordinals: 5TH → 5th, 1ST → 1st
    pattern = r'^\d+(ST|ND|RD|TH)$'
    if re.match(pattern, normalized_name):
        # Extract number and suffix
        match_obj = re.match(r'^(\d+)(ST|ND|RD|TH)$', normalized_name)
        if match_obj:
            num = match_obj.group(1)
            suffix = match_obj.group(2).lower()  # Convert TH → th
            normalized_name = f"{num}{suffix}"
    
    # Normalize street type (AV → Av, ST → St, etc.)
    # Alameda uses title case
    type_map = {
        'ST': 'St', 'AV': 'Av', 'AVE': 'Av', 'AVENUE': 'Av',
        'WY': 'Wy', 'WAY': 'Wy',
        'BL': 'Bl', 'BLVD': 'Bl', 'BOULEVARD': 'Bl',
        'RD': 'Rd', 'ROAD': 'Rd',
        'DR': 'Dr', 'DRIVE': 'Dr',
        'PL': 'Pl', 'PLACE': 'Pl',
        'CT': 'Ct', 'COURT': 'Ct',
        'LN': 'Ln', 'LANE': 'Ln',
        'SQ': 'Sq', 'SQUARE': 'Sq',
        'TE': 'Te', 'TER': 'Te', 'TERRACE': 'Te',
        'CI': 'Ci', 'CIR': 'Ci', 'CIRCLE': 'Ci',
    }
    
    street_type_upper = street_type.upper()
    normalized_type = type_map.get(street_type_upper, street_type.title())
    
    # Build normalized address
    normalized = f"{street_num} {normalized_name} {normalized_type}"
    
    return normalized


def load_lookup_table(lookup_file):
    """Load the address lookup table"""
    if not os.path.exists(lookup_file):
        raise FileNotFoundError(f"Lookup table not found: {lookup_file}")
    
    df = pd.read_csv(lookup_file)
    return df


def geocode_from_lookup(address, lookup_file='/Users/johngage/berkeley-data/alameda_lookup_complete.csv'):
    """
    Geocode an address using the lookup table
    Handles: FIFTH → 5th, case conversion, float street numbers
    """
    # Normalize the input address to Alameda format
    normalized = normalize_address_for_lookup(address)
    
    if not normalized:
        return None
    
    # Load lookup table
    df_lookup = pd.read_csv(lookup_file)
    
    # Search for exact match in original_address
    match = df_lookup[df_lookup['original_address'] == normalized]
    
    if len(match) > 0:
        return {
            'latitude': float(match.iloc[0]['latitude']),
            'longitude': float(match.iloc[0]['longitude']),
            'apn': str(match.iloc[0]['APN']) if pd.notna(match.iloc[0]['APN']) else None,
            'original_address': match.iloc[0]['original_address'],
            'normalized_input': normalized
        }
    
    return None


def geocode_dataframe(df, address_column='address_display', 
                     lookup_file='/Users/johngage/berkeley-data/alameda_lookup_complete.csv'):
    """Geocode all addresses in a dataframe"""
    if 'latitude' not in df.columns:
        df['latitude'] = None
    if 'longitude' not in df.columns:
        df['longitude'] = None
    
    geocoded_count = 0
    
    for idx, row in df.iterrows():
        if pd.notna(row.get('latitude')):
            continue
        
        address = row[address_column]
        result = geocode_from_lookup(address, lookup_file)
        
        if result:
            df.at[idx, 'latitude'] = result['latitude']
            df.at[idx, 'longitude'] = result['longitude']
            if 'APN' in df.columns:
                df.at[idx, 'APN'] = result['apn']
            geocoded_count += 1
    
    return df, geocoded_count


def validate_coordinates(latitude, longitude, city='Berkeley'):
    """Validate coordinates are within bounds"""
    if city == 'Berkeley':
        return (BERKELEY_BOUNDS['lat_min'] <= latitude <= BERKELEY_BOUNDS['lat_max'] and
                BERKELEY_BOUNDS['lon_min'] <= longitude <= BERKELEY_BOUNDS['lon_max'])
    return False


def add_manual_geocode(address, latitude, longitude, apn=None,
                       lookup_file='/Users/johngage/berkeley-data/alameda_lookup_complete.csv'):
    """Add a manually geocoded address to the lookup table"""
    if not validate_coordinates(latitude, longitude):
        print(f"WARNING: Coordinates outside Berkeley bounds")
        return False
    
    df_lookup = pd.read_csv(lookup_file)
    
    # Normalize address before adding
    normalized = normalize_address_for_lookup(address)
    
    new_entry = pd.DataFrame([{
        'original_address': normalized if normalized else address,
        'latitude': float(latitude),
        'longitude': float(longitude),
        'APN': str(apn) if apn else ''
    }])
    
    df_lookup = pd.concat([df_lookup, new_entry], ignore_index=True)
    df_lookup.to_csv(lookup_file, index=False)
    
    print(f"Added {address} to lookup table")
    return True


def get_unmatched_addresses(df, address_column='address_display'):
    """Get list of addresses without coordinates"""
    if 'latitude' not in df.columns:
        return df[address_column].tolist()
    
    unmatched = df[df['latitude'].isna()]
    return unmatched[address_column].tolist()


# Aliases for notebook compatibility
def geocode_address(address, lookup_file='/Users/johngage/berkeley-data/alameda_lookup_complete.csv'):
    return geocode_from_lookup(address, lookup_file)


def geocode_batch(df, address_column='address_display', 
                 lookup_file='/Users/johngage/berkeley-data/alameda_lookup_complete.csv'):
    return geocode_dataframe(df, address_column, lookup_file)


def validate_berkeley_coords(latitude, longitude):
    return validate_coordinates(latitude, longitude, city='Berkeley')


def manual_geocode_entry(address, latitude, longitude, apn=None,
                        lookup_file='/Users/johngage/berkeley-data/alameda_lookup_complete.csv'):
    return add_manual_geocode(address, latitude, longitude, apn, lookup_file)
'''

# Write the module
with open('/Users/johngage/berkeley-data/modules/geocoder.py', 'w') as f:
    f.write(complete_module)

print("✅ Updated geocoder.py (fixed regex escapes)")
print("\nRestart kernel, then test!")

✅ Updated geocoder.py (fixed regex escapes)

Restart kernel, then test!


In [10]:
# CELL: Check Current Geocoder Module

with open('/Users/johngage/berkeley-data/modules/geocoder.py', 'r') as f:
    current_content = f.read()

print("📄 Current geocoder.py content:")
print("="*70)
print(current_content[:500])  # First 500 chars
print("\n... (truncated)")
print(f"\nTotal length: {len(current_content)} characters")

# Check what functions are defined
import re
functions = re.findall(r'^def (\w+)\(', current_content, re.MULTILINE)
print(f"\nFunctions found: {functions}")

📄 Current geocoder.py content:
"""
Geocoding module for Berkeley housing project addresses
Uses Alameda County address lookup table with proper normalization
"""

import pandas as pd
import os
import re


# Berkeley bounds constant
BERKELEY_BOUNDS = {
    'lat_min': 37.84,
    'lat_max': 37.91,
    'lon_min': -122.32,
    'lon_max': -122.23
}


def normalize_street_name(name):
    """
    Convert between word and number forms of street names
    FIFTH ↔ 5th, SIXTH ↔ 6th, etc.
    """
    # Uppercase for consistency
    name_u

... (truncated)

Total length: 7352 characters

Functions found: ['normalize_street_name', 'normalize_address_for_lookup', 'load_lookup_table', 'geocode_from_lookup', 'geocode_dataframe', 'validate_coordinates', 'add_manual_geocode', 'get_unmatched_addresses', 'geocode_address', 'geocode_batch', 'validate_berkeley_coords', 'manual_geocode_entry']


## 3. Berkeley Bounds Validation

In [3]:
# Display Berkeley bounds
print("Berkeley City Bounds:")
print(f"  Latitude:  {BERKELEY_BOUNDS['lat_min']} to {BERKELEY_BOUNDS['lat_max']}")
print(f"  Longitude: {BERKELEY_BOUNDS['lon_min']} to {BERKELEY_BOUNDS['lon_max']}")

# Test validation
test_coords = [
    (37.87, -122.27, "Downtown Berkeley"),
    (37.90, -122.26, "North Berkeley"),
    (37.80, -122.30, "Outside Berkeley"),
]

print("\nCoordinate Validation Tests:")
for lat, lon, desc in test_coords:
    valid = validate_berkeley_coords(lat, lon)
    status = "VALID" if valid else "INVALID"
    print(f"  {desc}: ({lat}, {lon}) -> {status}")

Berkeley City Bounds:
  Latitude:  37.84 to 37.91
  Longitude: -122.32 to -122.23

Coordinate Validation Tests:
  Downtown Berkeley: (37.87, -122.27) -> VALID
  North Berkeley: (37.9, -122.26) -> VALID
  Outside Berkeley: (37.8, -122.3) -> INVALID


## 4. Test Geocoding Original

In [4]:
# Test single address geocoding
test_addresses = [
    "2660 BANCROFT Way",
    "130 BERKELEY Sq",
    "1850 BERRYMAN St",
    "5 W PARNASSUS Ct",
    "0 LE ROY Ave	",
    "1463 Le Roy Ave",
    "1473 Le Roy Ave",
    "2700 SHATTUCK Ave",
    "1914 FIFTH St",
    "1914 5TH Street",
    "2200 FIFTH St",
    "1618 SIXTH St",
    "2276 Shattuck Avenue",
    "9999 Fake Street",  # Should not match
]

print("Geocoding Tests:")
print("="*70)

for addr in test_addresses:
    result = geocode_address(addr, lookup_df)
    
    if result['success']:
        print(f"\n{addr}")
        print(f"  Matched: {result['matched_address']}")
        print(f"  Coords: ({result['latitude']:.6f}, {result['longitude']:.6f})")
        print(f"  APN: {result['apn']}")
    else:
        print(f"\n{addr}")
        print(f"  NO MATCH: {result.get('error', 'Unknown error')}")

Geocoding Tests:


TypeError: stat: path should be string, bytes, os.PathLike or integer, not DataFrame

In [6]:
# CELL: Test Geocoding - Complete Test Suite (FIXED)

import sys
sys.path.insert(0, '/Users/johngage/berkeley-data')

# Force fresh import
if 'modules.geocoder' in sys.modules:
    del sys.modules['modules.geocoder']

from modules.geocoder import (
    geocode_address,
    geocode_batch,
    validate_berkeley_coords,
    get_unmatched_addresses,
    BERKELEY_BOUNDS
)

print("✅ All imports successful!\n")
print("="*70)
print("\n1️⃣ TEST: Individual Address Geocoding\n")

# Test addresses from your projects
test_addresses = [
    '1914 FIFTH St',      # Numbered street (word form)
    '2200 FIFTH St',      # Another numbered street
    '1914 fifth st',      # Test case insensitivity
    '2700 SHATTUCK Ave',  # Regular address
    '130 BERKELEY Sq',    # Special location
]

for addr in test_addresses:
    result = geocode_address(addr)
    if result:
        coords_valid = validate_berkeley_coords(result['latitude'], result['longitude'])
        status = "✅ VALID" if coords_valid else "⚠️ OUTSIDE BOUNDS"
        print(f"{status} {addr:25} → {result['latitude']:.6f}, {result['longitude']:.6f}")
    else:
        print(f"❌ FAILED {addr:25} → Not found in lookup")

print("\n" + "="*70)
print("\n2️⃣ TEST: Batch Geocoding on Projects File\n")

import pandas as pd

# Load your projects
df = pd.read_csv('/Users/johngage/berkeley-data/housing_projects_FINAL.csv')

print(f"Loaded: {len(df)} projects")
print(f"Missing coordinates before: {df['latitude'].isna().sum()}")

# Get unmatched addresses
unmatched_before = get_unmatched_addresses(df, 'address_display')
print(f"Addresses needing geocoding: {len(unmatched_before)}\n")

if unmatched_before:
    print("Addresses to geocode:")
    for addr in unmatched_before[:10]:  # Show first 10
        print(f"   • {addr}")
    if len(unmatched_before) > 10:
        print(f"   ... and {len(unmatched_before) - 10} more")

# Batch geocode
print("\n" + "="*70)
print("\n3️⃣ RUNNING BATCH GEOCODING...\n")

df_geocoded, count = geocode_batch(df, 'address_display')

print(f"✅ Geocoded: {count} addresses")
print(f"   Still missing: {df_geocoded['latitude'].isna().sum()}")

# Get remaining unmatched
unmatched_after = get_unmatched_addresses(df_geocoded, 'address_display')

if unmatched_after:
    print(f"\n⚠️ Remaining addresses without coordinates ({len(unmatched_after)}):")
    for addr in unmatched_after:
        print(f"   • {addr}")

# Save results
df_geocoded.to_csv('/Users/johngage/berkeley-data/housing_projects_FINAL.csv', index=False)
print(f"\n✅ Saved updated file")

print("\n" + "="*70)
print("\n4️⃣ FINAL STATISTICS\n")

total = len(df_geocoded)
with_coords = df_geocoded['latitude'].notna().sum()
match_rate = (with_coords / total) * 100

print(f"Total projects: {total}")
print(f"With coordinates: {with_coords}")
print(f"Match rate: {match_rate:.1f}%")

if match_rate == 100:
    print(f"\n🎉 PERFECT! ALL {total} PROJECTS GEOCODED! 🎉")
elif match_rate >= 98:
    print(f"\n✅ Excellent! Only {total - with_coords} addresses need manual geocoding")
else:
    print(f"\n⚠️ Still need to geocode {total - with_coords} addresses")

print("\n" + "="*70)
print("\n5️⃣ BERKELEY BOUNDS VALIDATION\n")

print(f"Berkeley bounds:")
print(f"  Latitude:  {BERKELEY_BOUNDS['lat_min']} to {BERKELEY_BOUNDS['lat_max']}")
print(f"  Longitude: {BERKELEY_BOUNDS['lon_min']} to {BERKELEY_BOUNDS['lon_max']}")

# Check if any coordinates are outside bounds
geocoded_projects = df_geocoded[df_geocoded['latitude'].notna()]
outside_bounds = []

for idx, row in geocoded_projects.iterrows():
    if not validate_berkeley_coords(row['latitude'], row['longitude']):
        outside_bounds.append(row['address_display'])

if outside_bounds:
    print(f"\n⚠️ WARNING: {len(outside_bounds)} addresses have coordinates outside Berkeley:")
    for addr in outside_bounds:
        print(f"   • {addr}")
else:
    print(f"\n✅ All geocoded addresses are within Berkeley bounds!")

print("\n" + "="*70)

✅ All imports successful!


1️⃣ TEST: Individual Address Geocoding

❌ FAILED 1914 FIFTH St             → Not found in lookup
❌ FAILED 2200 FIFTH St             → Not found in lookup
❌ FAILED 1914 fifth st             → Not found in lookup
✅ VALID 2700 SHATTUCK Ave         → 37.859780, -122.267828
❌ FAILED 130 BERKELEY Sq           → Not found in lookup


2️⃣ TEST: Batch Geocoding on Projects File

Loaded: 84 projects
Missing coordinates before: 11
Addresses needing geocoding: 11

Addresses to geocode:
   • 1914 FIFTH St
   • 2200 FIFTH St
   • 1618 SIXTH St
   • 2221 FIFTH St
   • 1650 FIFTH St
   • 1464 SIXTH St
   • 2812 EIGHTH St
   • 1716 SEVENTH St
   • 2808 NINTH St
   • 0 LE ROY Ave
   ... and 1 more


3️⃣ RUNNING BATCH GEOCODING...

✅ Geocoded: 0 addresses
   Still missing: 11

⚠️ Remaining addresses without coordinates (11):
   • 1914 FIFTH St
   • 2200 FIFTH St
   • 1618 SIXTH St
   • 2221 FIFTH St
   • 1650 FIFTH St
   • 1464 SIXTH St
   • 2812 EIGHTH St
   • 1716 SEVENTH St

## 5. Load Housing Projects

In [48]:
# Load housing projects
housing_path = Path(CONFIG['paths']['housing_projects'])

if housing_path.exists():
    df_projects = load_csv(housing_path)
    print(f"Loaded {len(df_projects)} projects")
    
    # Check existing coordinates
    if 'latitude' in df_projects.columns:
        has_coords = df_projects['latitude'].notna().sum()
        print(f"Already geocoded: {has_coords} ({100*has_coords/len(df_projects):.1f}%)")
    
    # Show sample
    display(df_projects[['address_display', 'latitude' ,'net_units', 'status']].head(54))
else:
    print(f"File not found: {housing_path}")
    print("\nAvailable CSV files:")
    for f in DATA_DIR.glob('*.csv'):
        print(f"  {f.name}")

Loaded 84 rows from housing_projects_FINAL.csv
Loaded 84 projects
Already geocoded: 73 (86.9%)


,address_display,latitude,net_units,status
0,1750 SACRAMENTO St,37.874312,739.0,Under Review
1,2276 SHATTUCK Ave,37.867738,336.0,In Review
2,2700 SHATTUCK Ave,37.859780,276.0,Corrections Pending Applicant
3,1914 FIFTH St,NaN,257.0,Under Review
4,2425 DURANT Ave,37.867951,250.0,Pending Final Action
5,2029 UNIVERSITY Ave,37.872262,240.0,Pending Final Action
6,2601 SAN PABLO Ave,37.859268,223.0,In Review
7,2920 SHATTUCK Ave,37.856347,221.0,In Review
8,1899 OXFORD St,37.874496,212.0,Pending Final Action
9,3000 SHATTUCK Ave,37.855008,166.0,In Review


## 6. Batch Geocoding

In [ ]:
# Geocode all addresses
if 'df_projects' in dir():
    # Get addresses to geocode
    address_col = 'address_display' if 'address_display' in df_projects.columns else 'address'
    addresses = df_projects[address_col].tolist()
    
    print(f"Geocoding {len(addresses)} addresses...")
    print("="*60)
    
    # Batch geocode
    results_df = geocode_batch(addresses, lookup_df, progress=True)
    
    # Merge results back to projects
    df_projects_geocoded = df_projects.copy()
    df_projects_geocoded['latitude_new'] = results_df['latitude'].values
    df_projects_geocoded['longitude_new'] = results_df['longitude'].values
    df_projects_geocoded['apn_geocoded'] = results_df['apn'].values
    df_projects_geocoded['geocode_success'] = results_df['success'].values
    
    # Use new coords where available
    mask = results_df['success'].values
    df_projects_geocoded.loc[mask, 'latitude'] = df_projects_geocoded.loc[mask, 'latitude_new']
    df_projects_geocoded.loc[mask, 'longitude'] = df_projects_geocoded.loc[mask, 'longitude_new']

## 7. Review Unmatched Addresses

In [ ]:
# Get unmatched addresses for manual review
if 'df_projects_geocoded' in dir():
    unmatched = df_projects_geocoded[~df_projects_geocoded['geocode_success']].copy()
    
    if len(unmatched) > 0:
        print(f"Unmatched addresses: {len(unmatched)}")
        print("="*60)
        
        # Show unmatched
        cols = ['address_display', 'net_units', 'status']
        cols = [c for c in cols if c in unmatched.columns]
        display(unmatched[cols])
        
        # Export for manual review
        unmatched_path = DATA_DIR / 'unmatched_addresses.csv'
        unmatched[cols].to_csv(unmatched_path, index=False)
        print(f"\nExported to: {unmatched_path}")
    else:
        print("All addresses matched!")

## 8. Manual Geocoding (for unmatched addresses)

**TODO:** Add coordinates for unmatched addresses manually.

In [47]:
# TODO: Manual geocoding for unmatched addresses
# Use Google Maps, OpenStreetMap, or Alameda County GIS to find coordinates

# Example:
# manual_geocode_entry(
#     address="123 Example St",
#     latitude=37.870,
#     longitude=-122.270,
#     apn="123-456-789"
# )

# After adding manual entries, re-run geocoding:
# results_df = geocode_batch(addresses, lookup_df)

# print("Manual geocoding entries can be added here.")
# print("Use Google Maps or Alameda County GIS to find coordinates.")
# CELL: Manual Geocoding Entry


# TODO: Add your manual coordinates here
manual_coords = {
    # Format: 'ADDRESS': (latitude, longitude)
    # Get coordinates from Google Maps (right-click location → click coordinates)
    
    # Example (replace with real coordinates):
    # '1914 FIFTH St': (37.870123, -122.268456),










    
}

# Apply manual coordinates
if manual_coords:
    for address, (lat, lon) in manual_coords.items():
        mask = df['address_display'] == address
        
        if mask.any():
            # Validate coordinates are in Berkeley
            if 37.84 < lat < 37.91 and -122.32 < lon < -122.23:
                df.loc[mask, 'latitude'] = lat
                df.loc[mask, 'longitude'] = lon
                print(f"✅ Added: {address}")
                print(f"   Coords: {lat:.6f}, {lon:.6f}")
            else:
                print(f"⚠️ WARNING: {address} coordinates outside Berkeley bounds!")
                print(f"   Coords: {lat:.6f}, {lon:.6f}")
                print(f"   Expected: lat 37.84-37.91, lon -122.32 to -122.23")
        else:
            print(f"❌ Address not found: {address}")
    
    # Show updated count
    still_missing = df['latitude'].isna().sum()
    print(f"\n📊 Still missing coordinates: {still_missing}/{len(df)}")
else:
    print("⚠️ No manual coordinates entered yet")
    print("   Add coordinates in the manual_coords dictionary above")

⚠️ No manual coordinates entered yet
   Add coordinates in the manual_coords dictionary above


## 9. Export Geocoded Data

In [ ]:
# Export geocoded projects
if 'df_projects_geocoded' in dir():
    # Clean up temporary columns
    cols_to_drop = ['latitude_new', 'longitude_new', 'geocode_success']
    cols_to_drop = [c for c in cols_to_drop if c in df_projects_geocoded.columns]
    df_final = df_projects_geocoded.drop(columns=cols_to_drop)
    
    # Summary stats
    total = len(df_final)
    with_coords = df_final['latitude'].notna().sum()
    
    print(f"Geocoding Summary:")
    print(f"  Total projects: {total}")
    print(f"  With coordinates: {with_coords} ({100*with_coords/total:.1f}%)")
    print(f"  Missing coordinates: {total - with_coords}")
    
    # Export
    output_path = DATA_DIR / 'housing_projects_geocoded.csv'
    df_final.to_csv(output_path, index=False)
    print(f"\nSaved: {output_path}")

---

## Summary

This notebook:
- Loaded Alameda County address lookup table
- Geocoded housing projects using lookup
- Validated coordinates within Berkeley bounds
- Identified unmatched addresses for manual review

**Next:** Run `B1_lifecycle_tracking.ipynb` to link permit stages.